In [1]:
import plotly.graph_objects as go

import os
os.environ["SHAPE_RESTORE_SHX"] = "YES"

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.features import geometry_mask
from pathlib import Path
import re
import gc


import winsound

In [2]:

mspa_classes = {
    1: "Core", 2: "Small Islet", 3: "Perforation", 4: "Edge", 0: "No Forest", 6: "Medium Islet", 7: "Big Islet",
}

class_colors = {
    "Core": "#006400",
    "Small Islet": "#FFA07A",
    "Perforation": "#905590",
    "Edge": "#90EE90",
    "No Forest": "#B3B3B3",
    "Medium Islet": "#F0E68C" ,
    "Big Islet":  "#808000",
}

# Sankey Plot - Argentina

In [3]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Argentina_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986,  2005,  2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
traj_df.to_csv(folder / "MSPA_trajectory_Argentina_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.0f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.0f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.0f}K"
    else:
        return f"{sign}{n:.0f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory ({years}) - Argentina",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_Argentina_vislets_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_Argentina.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

Using years: [1986, 2005, 2023]
Collected 83 unique (hop, from, to) transitions
    hop  source_class  target_class               source               target  \
0     0          Core          Edge          Core (1986)          Edge (2005)   
1     0          Core    Deforested          Core (1986)    Deforested (2005)   
2     0          Edge   Small Islet          Edge (1986)   Small Islet (2005)   
3     0          Edge          Edge          Edge (1986)          Edge (2005)   
4     0          Edge    Deforested          Edge (1986)    Deforested (2005)   
5     1   Small Islet          Edge   Small Islet (2005)          Edge (2023)   
6     1          Edge          Core          Edge (2005)          Core (2023)   
7     1          Edge   Perforation          Edge (2005)   Perforation (2023)   
8     1          Edge          Edge          Edge (2005)          Edge (2023)   
9     1          Edge    Deforested          Edge (2005)    Deforested (2023)   
10    0          Core        

# Sankey Plot - Brazil

In [ ]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Brazil_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986,  2005,  2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "MSPA_trajectory_Argentina_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.0f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.0f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.0f}K"
    else:
        return f"{sign}{n:.0f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory ({years}) - Brazil",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_Brazil_vislets_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_Brazil.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

Using years: [1986, 2005, 2023]


# Sankey Plot - Paraguay

In [ ]:
folder = Path(r"C:\Users\atrumper_paper3")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Paraguay_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986,  2005,  2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "MSPA_trajectory_Argentina_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.0f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.0f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.0f}K"
    else:
        return f"{sign}{n:.0f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory ({years}) - Paraguay",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_Paraguay_vislets_v0311.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_Paraguay.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

# Sankey Plot Alto Parana Ecoregion

In [ ]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\alto_parana_3857.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986,  2005,  2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "MSPA_trajectory_Argentina_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.0f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.0f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.0f}K"
    else:
        return f"{sign}{n:.0f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory ({years}) - APAF ecoregion",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_APAFecoregion_vislets_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_APAFecoregion.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

# Sankey Study Area APAF

In [7]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Study_area\AltoParana_adm_bound_50km_50perc_v3857.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986,  2005,  2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "MSPA_trajectory_Argentina_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.0f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.0f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.0f}K"
    else:
        return f"{sign}{n:.0f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory ({years}) - APAF study area",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_APAFstudyarea_vislets_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_APAFstudyarea.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

Collected 84 unique (hop, from, to) transitions
    hop  source_class  target_class               source               target  \
0     0   Small Islet   Small Islet   Small Islet (1986)   Small Islet (2005)   
1     0   Small Islet    Deforested   Small Islet (1986)    Deforested (2005)   
2     0   Perforation  Medium Islet   Perforation (1986)  Medium Islet (2005)   
3     0   Perforation    Deforested   Perforation (1986)    Deforested (2005)   
4     0          Edge   Small Islet          Edge (1986)   Small Islet (2005)   
5     0          Edge  Medium Islet          Edge (1986)  Medium Islet (2005)   
6     0          Edge    Deforested          Edge (1986)    Deforested (2005)   
7     1   Small Islet   Small Islet   Small Islet (2005)   Small Islet (2023)   
8     1   Small Islet    Deforested   Small Islet (2005)    Deforested (2023)   
9     1  Medium Islet  Medium Islet  Medium Islet (2005)  Medium Islet (2023)   
10    1  Medium Islet    Deforested  Medium Islet (2005)    D